# 黑马头条项目｜第11天：实时计算热点文章与 Kafka Streams

> 学习目标：理解流式计算与 Kafka Streams，掌握 KStream、窗口、状态存储和 Spring Boot 集成；将用户行为发送到 Kafka，按文章聚合后更新文章统计与热点缓存。

本篇根据百度网盘 `day11-实时计算热点文章、kafkaStream/讲义/新热文章-实时计算.md` 网页预览整理，衔接第09天行为记录和第10天定时计算热点。讲义文件名为“新热文章”，正文标题为“热点文章-实时计算”。补充资料目录页面显示“没有数据”，本篇依据完整讲义文本及其中的代码片段编写。

本篇保留原消息字段、枚举、Topic、窗口长度、状态库名称和计算步骤。“工程补充”“修正示意”用于解释课程代码的边界，不代表参考工程已实现。未读取代码目录、PPT、视频或图片资源包；未启动 Kafka、Streams、MySQL、Redis、Nacos 或课程微服务，未调用真实行为接口。图片没有嵌入，连接地址均为占位符。

资料：[黑马头条项目分享链接](https://pan.baidu.com/s/1u7PNExeoEsYGh5pAnSyocg?pwd=9987)。


## 1. 从每天计算到持续更新

第10天按天计算文章分值、缓存每频道和默认推荐的 Top 30。这样查询快，但两次任务之间的新行为不会及时改变热点排名。第11天引入 Kafka Streams，对持续到来的行为做增量处理。

| 维度 | 第10天定时计算 | 第11天实时处理 |
|---|---|---|
| 输入 | 查询时间范围内的文章统计 | 持续接收行为变化消息 |
| 触发 | XXL-JOB定时任务 | Kafka记录到达、窗口聚合 |
| 计算方式 | 批量计算、排序、重建Top30 | 更新文章统计、重算单篇分值、尝试替换Top30 |
| 主要作用 | 初始化、周期校准 | 降低热点变化延迟 |
| 正确性重点 | 查询窗口、整批缓存发布 | 去重、窗口语义、并发更新、消息与外部存储一致性 |

“实时”不代表没有延迟。网络、分区排队、窗口、状态缓存提交和消费端处理都会影响结果可见时间。讲义的10秒窗口也不等于保证每10秒只输出一个最终结果。

工程补充：可将批量任务作为校准，流式更新作为增量加速，但两条链路必须使用一致的计数口径、权重和版本规则。不能让凌晨任务覆盖一份更晚的实时结果而没有边界。


## 2. 流式计算与 Kafka Streams

流式计算的输入和输出在时间上持续、无界，无法等全部数据到齐后才开始。先定义计算逻辑，记录到达后执行，通常用增量计算提高效率。日志分析、实时看板、公交位置、文章热度都属于常见场景。

Kafka Streams 是 Java 客户端库，可嵌入普通 Java/Spring Boot 应用，消费 Kafka记录、执行拓扑并把结果写回 Kafka。它利用分区扩展任务，通过可恢复的 state store 保存聚合/窗口状态，提供 DSL 和 Processor API。

~~~text
Kafka输入Topic
       ↓ Source Processor（无上游）
解析 → 转换 → 分组 → 窗口 → 聚合状态
       ↓ Sink Processor（无下游）
Kafka输出Topic
~~~

资料提到事件时间、迟到数据和正好一次处理能力。它们需要正确的配置和拓扑设计；当前讲义配置并未展示 `processing.guarantee`，不能据此认为示例已启用 exactly-once。Kafka内部的处理保障也不会自动覆盖外部 MySQL/Redis。

“除Kafka外无外部依赖”是指无需另建流计算集群；有状态的聚合仍需要应用本地状态目录、存储空间和恢复资源。


## 3. KStream、KTable与聚合结果

讲义重点介绍 KStream：它把消息理解成不断追加的事件。相同 key 的新记录不是替换旧记录。

~~~text
输入 (alice, 1)，再输入 (alice, 3)
按KStream事件求和：1 + 3 = 4
~~~

为理解后面的 `.aggregate().toStream()`，补充 KTable 的区别：

| 概念 | 数据含义 |
|---|---|
| KStream | 追加事件，每条记录都代表一次输入 |
| KTable | 每个key的最新状态；新值可替换该key的旧值 |
| 窗口聚合表 | 每个“key+窗口”的最新累计结果 |

`count`、`aggregate` 产生聚合表，其 `toStream()` 是聚合状态的变更流，值可能先是1，再是2，再是3。它不自动转成每次新增1，也不自动等待窗口关闭。

这一区别决定数据库如何消费：累计快照应覆盖或按版本比较求差；真正独立的增量才可每次直接相加。后面的课程实现需要特别关注这一点。


## 4. 原生 Word Count 入门案例

讲义先用单词计数学习拓扑，输入例如 `hello kafka hello itcast`：

1. 从 `itcast-topic-input` 建立 KStream；
2. `flatMapValues` 按空格拆成单词；
3. `groupBy((key, value) -> value)` 按单词分组；
4. `windowedBy(TimeWindows.of(Duration.ofSeconds(10)))` 建10秒窗口；
5. `count()` 统计；
6. `toStream()` 将聚合变更转回流；
7. 将窗口key取成单词、计数转成字符串，写 `itcast-topic-out`。

~~~java
streamsBuilder.<String, String>stream("itcast-topic-input")
    .flatMapValues(value -> Arrays.asList(value.split(" ")))
    .groupBy((key, word) -> word)
    .windowedBy(TimeWindows.of(Duration.ofSeconds(10)))
    .count()
    .toStream()
    .map((windowedWord, count) ->
        new KeyValue<>(windowedWord.key(), count.toString()))
    .to("itcast-topic-out");
~~~

这是讲义拓扑的精简表示，省略 imports、创建配置和启动。需要 String key/value Serde，`application.id` 原生案例为 `streams-quickstart`，Kafka地址用 `<KAFKA_BOOTSTRAP_SERVERS>`。

讲义代码使用连字符Topic `itcast-topic-input/out`，测试文字却写下划线 `itcast_topic_input/out`。这些是不同Topic，联调时必须统一。字数也可能多次输出窗口累计变化，不能把“汇总成一条”当作严格输出保证。

生产还应处理空白、标点、大小写与空词；本案例只用于理解算子。


## 5. Spring Boot 集成与版本

讲义在 Kafka demo 工程引入 `org.apache.kafka:kafka-streams`，并排除 `connect-json`、`kafka-clients` 两项依赖，未在片段中单独给出版本。实际依赖应与 Spring Boot、Spring Kafka、Kafka客户端及Broker版本对齐；排除后仍需其他依赖提供兼容客户端，不能直接猜测缺失的版本号。

`KafkaStreamConfig` 使用：

- `@Configuration`、`@EnableKafkaStreams` 开启支持；
- `@ConfigurationProperties(prefix="kafka")` 绑定 hosts/group；
- 按 `KafkaStreamsDefaultConfiguration.DEFAULT_STREAMS_CONFIG_BEAN_NAME` 注册 `KafkaStreamsConfiguration`；
- 设置 bootstrap servers、application.id、client.id、重试和默认 String Serde；
- 将 `StreamsBuilder` 注入 `@Bean` 方法定义拓扑，由 Spring 管理生命周期。

~~~yaml
kafka:
  hosts: <KAFKA_BOOTSTRAP_SERVERS>
  group: ${spring.application.name}
~~~

原配置拼接 `group + "_stream_aid"` 为 application.id，`group + "_stream_cid"` 为 client.id，设置重试10次。示例声明的 `MAX_MESSAGE_SIZE` 没有在展示代码中应用，不应说它已限制了消息大小。

原生客户端需自行 `start()`/关闭；Spring集成一般由框架生命周期管理。不要在 Spring 已管理的拓扑之外又启动同一份KafkaStreams实例。


## 6. application.id、分区与状态存储（工程补充）

| 配置/对象 | 作用与边界 |
|---|---|
| application.id | 标识同一流应用，关联消费组、内部Topic和状态恢复 |
| client.id | 客户端观测标识，不等同业务幂等键 |
| key Serde/value Serde | 把key/value序列化为Kafka记录，并正确还原 |
| 输入分区 | 决定记录可并行处理的范围，顺序只在分区内保证 |
| state store | 保存文章窗口累计值，可通过changelog恢复 |
| repartition Topic | key改变后，让相同分组key的数据汇集到正确任务 |

同一流应用多个实例通常共享稳定的 application.id，通过任务/分区分配分工；给每个副本随机应用ID可能让每份应用各自处理全量数据。环境隔离和不同业务拓扑则应使用不同ID。

本课先把无业务key的行为消息转为 articleId key，再进行分组。key变化后，Streams可能建立重新分区Topic；要考虑创建权限、分区数、磁盘和网络成本。Producer一开始就用 articleId 作 key有利于文章级路由，但仍需按实际拓扑确认是否发生重新分区。

不要随意改 application.id、状态库名或删除本地状态以“修复重复计数”：它们会影响历史重放、恢复和内部Topic，需有迁移与恢复方案。


## 7. 实时热点完整链路与原始名称

~~~text
用户点赞/取消/阅读等行为
       ↓ behavior服务保存个人Redis记录
UpdateArticleMess(type, articleId, add)
       ↓ hot.article.score.topic
article服务 HotArticleStreamHandler
       ↓ 按articleId分组，10秒窗口内按四类计数累加
state store: hot-atricle-stream-count-001
       ↓ 格式化ArticleVisitStreamMess
hot.article.incr.handle.topic
       ↓ ArticleIncrHandleListener
ApArticleService.updateScore()
       ├── 更新ap_article行为数量
       ├── 重新计算score，讲义额外乘3
       ├── 更新文章所属频道Top30
       └── 更新__all__推荐Top30
~~~

| 原始对象 | 名称 |
|---|---|
| 行为输入常量 | `HOT_ARTICLE_SCORE_TOPIC` |
| 输入Topic | `hot.article.score.topic` |
| 聚合输出常量 | `HOT_ARTICLE_INCR_HANDLE_TOPIC` |
| 输出Topic | `hot.article.incr.handle.topic` |
| 状态库 | `hot-atricle-stream-count-001`（原文拼写） |
| 拓扑类 | `HotArticleStreamHandler` |
| 监听器 | `ArticleIncrHandleListener` |
| 更新服务方法 | `updateScore(ArticleVisitStreamMess mess)` |

这是一条内部处理链，沿用第09天行为HTTP接口，不是新增一套“实时计算”前端接口。评论/收藏列在消息枚举与聚合字段中，但讲义完整Producer代码只展示点赞和阅读，不声称其余两类已在资料中完整实现。


## 8. 输入消息：UpdateArticleMess

讲义定义：

~~~java
@Data
public class UpdateArticleMess {
    private UpdateArticleType type;
    private Long articleId;
    private Integer add; // 修改增量，可正可负

    public enum UpdateArticleType {
        COLLECTION, COMMENT, LIKES, VIEWS;
    }
}
~~~

点赞消息结构示例：

~~~json
{
  "type": "LIKES",
  "articleId": 1001,
  "add": 1
}
~~~

取消点赞对应 `add: -1`；阅读示例为 `type: "VIEWS", add: 1`。这是行为数量的变化，不能直接当作分值；最终计数累加完成后才应用第10天的权重。

原消息没有 eventId、时间、对象类型或版本。工程扩展需补充去重与时间字段，并定义哪些消息是文章行为。第09天点赞type还能表示动态/评论；如果所有类型都套进文章统计，会误用目标ID，需要在发送端限定文章场景或明确消息对象类型。

Long ID在APP JSON方向按第09天字符串契约处理；内部Kafka消息的序列化也需统一，不应仅凭字段叫Long就认为所有消费者都能无损处理。


## 9. 点赞发送：关系变化与增量

讲义 `ApLikesBehaviorServiceImpl` 先检查 DTO和登录身份，再设置文章ID及 `LIKES` 类型：

1. operation=0：用 Redis Hash 查询 `<LIKE_BEHAVIOR><articleId>` 中 userId字段；已存在则返回“已点赞”；不存在则把 DTO JSON写入Hash，发送 `add=1`。
2. operation=1：删除 userId字段，发送 `add=-1`。
3. `kafkaTemplate.send(HOT_ARTICLE_SCORE_TOPIC, JSON.toJSONString(mess))` 发消息。

这给出了本课真实的 **Hash存储片段**。第09篇的Set方案是工程补充，不能把它说成课程源码使用的结构；前缀常量值本课没有定义。

工程注意点：

- 先查再写不原子，两个并发点赞都可能看到不存在并各发+1。
- 原取消分支没有检查是否真实删除，重复取消可能反复发送-1。
- `type`/`operation`为null时直接比较可能发生空指针，需先验证空值。
- 先写Redis再异步send，不保证发送成功；Redis写入与Kafka发送也没有同一事务。
- 本地 `@Transactional`不能将Redis和Kafka一起回滚。

修正原则：只有确认当前关系真正新增/删除时，才产生+1/-1事件。Redis关系变更可通过Lua等获得原子结果；跨系统事件可靠性还需另外设计，不能靠前端防连点解决。


## 10. 阅读发送：count与消息增量

讲义 `ApReadBehaviorServiceImpl` 要求登录。它按文章ID+userId查询Redis中的 `ReadBehaviorDto`：

- 已有记录时，将旧count与请求count相加；代码还将结果强转为short；
- 将当前DTO JSON写回Hash；
- 构造VIEWS消息，**固定 `add=1`**，而不是 `add=dto.count`；
- 异步发送到 `hot.article.score.topic`。

~~~text
已有个人count=5，本次请求count=2
Redis保存：7（按示例累加）
Kafka发送：VIEWS +1（示例固定值）
~~~

第09天接口只写“阅读次数”，语义未定义。第11天代码显示Redis分支按增量累加，但Kafka分支更像每次成功请求计一次，两者只有在每次count=1等约定下才一致。笔记据此指出差异，不宣称已联调验证。

工程补充：明确有效阅读口径；统一Redis与文章总计数使用的增量；限制count、避免short溢出；读改写可能丢失并发请求，应采用原子计数/受控更新；同一阅读事件重试需去重。

不能直接取已经加上旧值的 `dto.count` 发Kafka，否则会把个人累计值再次作为文章增量。必须在修改DTO前保存本次有效delta，或服务端统一按一次有效事件加1。


## 11. 按文章转换、分组与10秒窗口

讲义 `HotArticleStreamHandler` 的前半段：

~~~java
stream.map((key, value) -> {
    UpdateArticleMess mess = JSON.parseObject(value, UpdateArticleMess.class);
    return new KeyValue<>(mess.getArticleId().toString(),
                          mess.getType().name() + ":" + mess.getAdd());
})
.groupBy((key, value) -> key)
.windowedBy(TimeWindows.of(Duration.ofSeconds(10)));
~~~

输入JSON被转为 `(articleId字符串, 行为类型:增量)`，例如 `(1001, "LIKES:1")`。同一文章不同用户的行为会落到同一聚合key，在同一窗口里累计四类数据。

10秒窗口不是“每个文章从第一条消息开始独立倒计时10秒”。课程API表示按时间分段的窗口，边界依赖记录时间；原文未展示业务事件时间提取器或grace配置，具体默认值应结合Kafka Streams版本确认。

迟到消息、乱序事件和没有新消息时窗口是否关闭，都要理解 stream-time 的推进规则。仅墙钟过去10秒不一定立刻发出最终结果；应用本地状态缓存/commit也会影响可见频率。


## 12. aggregate的初始值、累加与state store

讲义使用字符串聚合器，初始值为：

~~~text
COLLECTION:0,COMMENT:0,LIKES:0,VIEWS:0
~~~

每到一条记录，它解析原累计值的四个整数，再根据新值的类型加上正/负增量，重新格式化字符串。结果保存到 `Materialized.as("hot-atricle-stream-count-001")`。

~~~text
同一文章、同一窗口：
LIKES:+1  → COLLECTION:0,COMMENT:0,LIKES:1,VIEWS:0
VIEWS:+1  → COLLECTION:0,COMMENT:0,LIKES:1,VIEWS:1
LIKES:-1  → COLLECTION:0,COMMENT:0,LIKES:0,VIEWS:1
~~~

聚合对象随后转成 `ArticleVisitStreamMess`：

~~~java
@Data
public class ArticleVisitStreamMess {
    private Long articleId;
    private int view;
    private int collect;
    private int comment;
    private int like;
}
~~~

字段是单数 `view/collect/comment/like`，不是数据库字段 `views/collection/likes`，需要明确映射。输出值代表窗口累计变化，不是整篇文章历史总量。

工程补充：类型化的计数对象比反复split字符串更容易扩展和校验，但需要显式Serde。要处理JSON为空、非法枚举、数值溢出等异常，避免一条坏消息导致流线程停止。


## 13. 关键问题：累计窗口结果不能反复当delta

原拓扑将聚合表 `.toStream()`，取 `windowedKey.key()` 只保留文章ID，再把四项累计结果发送给 `hot.article.incr.handle.topic`。后续数据库却每次把这些值加到历史计数。

这存在算重风险：

| 同一窗口的输入 | 可能输出的累计like | 若下游每次相加 |
|---|---:|---:|
| 第1次点赞+1 | 1 | +1 |
| 第2次点赞+1 | 2 | +2 |
| 第3次点赞+1 | 3 | +3 |
| 合计 | 窗口真实净增3 | 错误累加6 |

Kafka Streams缓存可能合并部分更新，所以输出不一定逐事件发生；但没有 `suppress` 或明确最终结果设计时，不能保证一个窗口只输出一次。反过来，把窗口开始/结束丢掉也让下游无法识别“这是哪个窗口的第几次修订”。

因此，讲义名为 `INCR_HANDLE_TOPIC` 并不能让累计快照自动变成独立增量。正确实现要先选择输出契约，保留窗口身份并做去重/版本控制；仅调整提交间隔无法根治。

这个结论来自已读取拓扑与更新代码的语义分析，并未运行参考工程；是否在其他未读代码中有额外保护，本篇没有依据断言。


## 14. 三种可选输出契约（工程补充）

| 方案 | 输出/消费方式 | 必须补齐的条件 |
|---|---|---|
| 独立事件delta | 每个有效行为事件只增量一次 | eventId去重、输入可靠性、数据库原子增量 |
| 最终窗口净增量 | 窗口关闭后输出一次四项结果，下游累计一次 | 关闭策略、grace、窗口ID、消费幂等、状态容量 |
| 窗口累计快照 | 可以输出多版，下游按版本记录最新值 | 窗口ID、单调版本、求差/重算、乱序保护 |

最终窗口方案可研究 `suppress(untilWindowCloses(...))`，但它是本篇补充，原讲义没有使用。须确认目标版本API、迟到策略、stream-time推进和buffer容量；不能把“加suppress”当作不考虑内存与幂等的完整修复。

窗口快照的扩展消息示意（非课程原字段）：

~~~json
{
  "articleId": "1001",
  "windowStart": "<WINDOW_START>",
  "windowEnd": "<WINDOW_END>",
  "revision": 2,
  "like": 2,
  "view": 1,
  "collect": 0,
  "comment": 0
}
~~~

消费端在事务中锁定“文章+窗口”记录；若revision更新则计算 `新快照 - 上次已应用快照`，更新文章计数并保存快照；同版或旧版跳过。窗口净增量可为负，不能将合法负快照拒绝为参数错误。

同样的最终窗口结果仍可能因重试重放而被再消费，需唯一键/业务去重记录。窗口聚合减少请求量，不能消除可靠性设计。


## 15. 聚合消息监听与文章统计更新

讲义 `ArticleIncrHandleListener` 用：

~~~java
@KafkaListener(topics = HotArticleConstants.HOT_ARTICLE_INCR_HANDLE_TOPIC)
public void onMessage(String value) {
    if (StringUtils.isNotBlank(value)) {
        ArticleVisitStreamMess mess =
            JSON.parseObject(value, ArticleVisitStreamMess.class);
        apArticleService.updateScore(mess);
    }
}
~~~

`updateScore` 先读取文章，更新阅读、收藏、评论、点赞数量，保存到数据库，再计算分值并更新两个热点列表。

**原文空值更新有问题**：

~~~java
// 原文形式：旧值为null时直接赋0，丢掉本次delta
article.setLikes(article.getLikes() == null
    ? 0 : article.getLikes() + mess.getLike());

// 修正示意：先将旧值规范为0，再加有效delta
article.setLikes((article.getLikes() == null ? 0 : article.getLikes())
    + mess.getLike());
~~~

四个计数都有这种原文结构。修正空值仍不解决并发读改写丢失、重复消息、乱序或累计窗口算重。

工程方向：数据库采用受控原子增量或乐观版本更新，在同一事务里记幂等处理结果；按文章确保合适的处理顺序。文章不存在、已删除/下架、计数为负、整数溢出等需要明确处理，不应盲目 `getById` 后解引用。


## 16. 分值计算与实时结果乘3

原文 `computeScore`沿用第10天公式：

~~~text
baseScore = views + likes×3 + comment×5 + collection×8
~~~

但第11天 `updateScore` 后面还有一行：

~~~java
Integer score = computeScore(apArticle);
score = score * 3;
~~~

因此，示例的实时缓存分值为基础分值的3倍，定时任务则直接写基础分值。资料没有解释这项差异的业务依据，不能断言它就是“新文章权重”或固定推荐策略。

~~~text
同一篇文章基础分=100
实时链路写300，定时链路写100
如果缓存同时含两种口径，文章之间无法公平比较
~~~

工程补充：若确需实时加权，应该明确触发条件、衰减/有效期，并对所有候选使用可比较的规则；否则统一使用一个score函数和版本。文章计数的单位、窗口和权重也必须与定时校准一致。

同时使用Integer运算存在溢出风险，乘3进一步扩大风险，应选择适合的数值类型和上限策略。


## 17. 课程Top30替换算法

讲义对文章所属频道和默认推荐各更新一次：

~~~text
hot_article_first_page_<channelId>
hot_article_first_page___all__
~~~

`replaceDataToRedis`先读JSON数组，再按以下分支修改：

1. 缓存存在文章：只更新该文章score。
2. 缓存不含文章、数量小于30：复制文章对象为HotArticleVo并加入。
3. 缓存不含文章、数量达到30：降序排序，找最低分；当前score更高时替换最低分文章。
4. 最后按score降序排序并 `SET` 整个数组。
5. 若缓存字符串为空，原文函数不初始化，也不写入。

~~~text
缓存含文章A？
    ├── 是 → 更新A分值
    └── 否 → 少于30？
              ├── 是 → 加入A
              └── 否 → A分值 > 最低分？是则替换
                                  ↓
                       排序后写回频道/默认列表
~~~

缓存为空时原示例依赖第10天初始化/定时重建，不能声称实时代码可从零建立列表。频道和默认列表是两次写入，也没有自动的跨key事务。


## 18. Top30算法的局限与缓存并发

这份算法是在当前30条上做候选替换，不是对所有文章维护完整、严格的Top30：

- 已在缓存的文章因取消行为降分时，只降低它的分值；原第31名不在当前列表，无法自动补位。
- 原文已有文章只更新score，不刷新likes/views/comment/collection等其他VO字段，展示计数可能滞后。
- 没有在更新前检查文章可见性、频道有效性、发布时间窗口。
- 同分候选使用严格 `>`，未定义次排序；已有文章重排也缺稳定次排序。
- 两个消费者同时读同一频道JSON，再分别SET，会丢掉其中一份更新。

同文章key分区只能帮助同文章顺序，**不同文章仍会并发写同一频道和默认推荐key**，因此不能靠articleId分区消除频道级覆盖。

工程补充方案：

- 为频道缓存使用版本条件更新、重试或合适的锁；
- 研究ZSet维护更大的候选池/完整范围，读取Top30；不能只存30项却承诺所有降分均可补位；
- 分数与文章元数据分别存储并统一查询可见性；
- 把默认推荐视为独立聚合维度，定义它与频道列表的一致性；
- 保留定时校准，修复漏消息、候选不足和失效文章。

这些方案需要兼顾原首页读取JSON的契约；换存储结构也必须修改读取实现，而不是只替换写入命令。


## 19. 消息可靠性与exactly-once边界

本课有三段需要区分：

| 环节 | 原文做法 | 工程风险 |
|---|---|---|
| Redis行为 → Kafka输入 | 先修改Hash，再异步send | 行为成功但发送失败；重试可能重复 |
| Kafka输入 → Streams状态/输出 | 窗口聚合并写输出Topic | 配置、状态恢复、异常记录影响语义 |
| Kafka输出 → MySQL/Redis | Listener读取、改计数、SET缓存 | 重放导致重复增量；数据库成功缓存失败 |

Kafka Producer可配置可靠发送与幂等，但Producer幂等不等于“重复HTTP请求只有一个业务事件”。应记录业务eventId、检查send结果，并有可恢复的事件存储/补发机制。

支持的版本可配置Streams正好一次处理，保障Kafka内部状态/offset/输出事务边界；它不会把MySQL和Redis纳入Kafka事务，也不修复“窗口快照被当delta”的业务契约错误。

消费端方向：事务内原子处理事件/窗口幂等记录与文章更新；明确offset提交时机，失败重试可重复执行而不重复计数；数据库成功后缓存刷新采用可重试读模型或补偿任务。坏消息应按错误类型进入隔离/死信处理，而不是持续重试卡住分区。


## 20. 定时与实时如何衔接（工程补充）

~~~text
有效行为 → 可靠事件 → 幂等统计更新 → 实时分值/候选更新
                                ↓
定时任务从同一统计口径重建 → 校准候选、修复过期数据
~~~

两条链路至少应约定：

1. 哪个数据源是文章计数的权威来源；Redis个人记录与文章表如何对账。
2. 阅读count、取消操作、评论删除等增量口径是否一致。
3. 定时窗口与实时更新是否使用相同候选范围；不能让无限旧文章通过实时消息重新入榜。
4. 取消实时结果额外乘3，或将确需的策略在两条链路统一配置。
5. 重建缓存时使用批次版本/切换边界，避免旧快照覆盖重建期间的新更新。
6. 文章上下架和删除事件对热点缓存的剔除路径。

只有周期重算不能补救已经重复累加到数据库的计数，因为它仍会从错误计数算分。计数对账/去重与Top30重建是两件事，要各自有修复方案。


## 21. 讲义代码检查表

| 原片段/现象 | 问题与处理方向 |
|---|---|
| Word Count代码连字符、测试文字下划线 | 不同Topic，发送/消费名称须统一 |
| Producer无显式articleId key | 文章初始路由未固定，后续分组需重新分区 |
| 点赞先查后写 | 并发可能双发+1；原子确认关系变化 |
| 取消无存在判断 | 重复取消可能多次-1；只有真实删除才发事件 |
| 阅读累加count却发固定+1 | 明确每请求计一次还是count增量，统一两端 |
| count强转short | 可能溢出，类型与范围需对齐 |
| 窗口聚合直接toStream | 可多次输出累计结果，不能逐条当独立delta |
| 输出key只取articleId | 丢窗口身份，无法按窗口去重/修订 |
| null ? 0 : old+delta | null分支丢本次delta，应先规范旧值再加 |
| 实时score乘3 | 与定时基础分不一致，须定义策略 |
| 只更新缓存score | 文章计数元数据可能陈旧 |
| Redis读整表改后SET | 并发覆盖，须控制版本/锁或改结构 |
| 空缓存直接跳过 | 需要定时初始化或明确兜底 |
| 已在榜文章降分 | 无榜外候选补位，不能保证严格Top30 |
| DTO包名/constants包名出现不同写法 | 原文类定义与import有差异，落地应按实际包路径统一 |

上述是对讲义可见片段的分析，未检查未读参考工程是否已有修正。


## 22. 联调与正确性测试计划（本次未执行）

### Streams入门

1. input/output Topic统一；发送 `hello kafka hello itcast`，同一窗口hello最终计数为2。
2. 观察输出是累计修订还是最终结果；不要假设只消费一条。
3. 同一词在两个窗口计数分开，验证输出保留窗口身份的补充方案。
4. 重启同application.id实例，检查状态恢复和消费进度。

### 行为与聚合

| 场景 | 需验证的结果 |
|---|---|
| 重复点赞/并发点赞 | 真实关系一次新增、事件净增1 |
| 取消两次 | 仅第一次真实删除发-1 |
| 阅读count不等于1 | Redis与文章计数使用同一有效增量口径 |
| 同窗口3次点赞 | 最终净增3，不能累计1+2+3为6 |
| +1后-1 | 窗口净增0，下游正确处理零/负变化 |
| 重复输出/旧窗口修订 | 版本/去重规则保护数据库计数 |
| 文章初始计数null | 本次+1后为1，而不是0 |
| 同文章并发消费 | 不丢增量，offset重放不重复应用 |
| 不同文章同频道更新 | 不覆盖另一篇的最新缓存结果 |
| 已入榜文章降分 | 按候选策略补位，或明确等定时校准 |
| 缓存空/文章下架 | 初始化/回退及剔除策略生效 |
| 定时重建同时有实时消息 | 分值与版本一致，不回退到旧结果 |

还要模拟Kafka发送失败、非法JSON、状态恢复、数据库成功缓存失败、消费后提交前宕机。确认错误不会被悄悄当成功或无限卡住分区。

这些是学习用计划；本次没有执行Kafka、Redis、MySQL或行为接口测试。


## 23. 监控与排查顺序

建议监控：Producer发送成功/失败、输入/输出记录数、分区积压、Streams线程状态、重新分区与state store大小、窗口延迟/丢弃、数据库增量处理/幂等跳过、Redis更新耗时、Top30最后刷新时间。

排查时按链路逐段定位：

1. HTTP用户行为成功了吗，个人Redis记录是否正确？
2. 输入Topic是否收到正确type/articleId/add，send结果是否确认？
3. article服务Streams拓扑是否启动，application.id和Serde是否一致？
4. 分组key与窗口身份正确吗，聚合输出是修订快照还是delta？
5. 输出Topic是否有消息，Listener是否有积压/异常？
6. 数据库计数是否空值丢增量、重复累加或并发覆盖？
7. 两个热点key是否更新，页面读的tag与字段是否一致？

计数翻倍先查累计窗口修订被当delta、消息重放和重复Producer事件；排名忽高忽低先查额外乘3、定时覆盖和并发缓存写入。日志保留event/window标识和必要业务ID，避免token、完整个人行为画像和正文泄露。


## 24. 复习与自测

~~~text
个人行为 → UpdateArticleMess增量 → Kafka输入
              ↓ 文章key + 10秒窗口 + state store
窗口变化 → ArticleVisitStreamMess → Kafka输出
              ↓ 幂等、正确的delta契约
文章计数 → 分值 → 频道/默认热点 → APP首页
~~~

1. **KStream里同key的1和3会替换吗？** 不会，两条独立事件求和为4。
2. **aggregate().toStream()会只输出一次窗口最终值吗？** 不保证，它输出聚合表变化；最终输出需要另外设计。
3. **为什么1、2、3不能都加到文章计数？** 它们可能是同窗口累计修订，真实净增3，直接相加会得到6。
4. **窗口消息为什么要保留起止时间？** 识别窗口、修订、重放和去重，单articleId不足以区分。
5. **阅读代码为什么需统一count？** Redis加请求count而Kafka固定+1；有效阅读口径必须一致。
6. **null ? 0 : old+delta有什么问题？** 旧计数为null时丢掉本次delta，应按0+delta处理。
7. **exactly-once会自动保护MySQL与Redis吗？** 不会，外部写入需幂等/事务/补偿；它也不修正错误的快照语义。
8. **已入榜文章降分能自动保持全局Top30吗？** 原算法没有榜外候选，不能保证；需完整候选或定时校准。
9. **为何同文章key不能解决频道缓存并发？** 不同文章也会写同一个频道/默认key，仍可能读改写覆盖。
10. **实时乘3是原需求明确要求吗？** 原代码确实乘3，但文档未说明依据，需与定时口径统一或定义策略。
11. **为什么每日重建不能修复算重的数据库计数？** 重建从已有计数算分，源计数错误会继续被使用。
12. **资料展示哪些行为Producer？** 完整示例是点赞和阅读；收藏、评论为枚举和聚合支持，未展示完整Producer实现。

### 本篇覆盖

| 已读取资料 | 覆盖内容 |
|---|---|
| 新热文章-实时计算.md | 流式计算、Kafka Streams、Word Count、Spring集成、行为Producer、窗口聚合、文章计数、热点替换、Listener |

已区分课程约定与工程补充：窗口幂等、可靠发送、候选补位、并发控制和监控均是额外说明。原工程、PPT、视频和图片未读取，项目环境及业务测试未执行。
